### Extract PDF

In [1]:
from pathlib import Path
import pymupdf
import os
import re
import hashlib
DOCUMENT_PATH = '../data/papers/'


In [2]:
def extract_pages(path: Path) -> list[dict]:
    
    with pymupdf.open(path) as doc:
        pages = []
        id = path.stem
        sha256 = hashlib.sha256(path.read_bytes()).hexdigest()
        for i, page in enumerate(doc):
            pages.append({
                'paper_id': id,
                'source_sha256': sha256,
                'page': i + 1,                
                'text': page.get_text()
            })
            
        return pages

In [3]:
def chunk_pages(pages: list[dict], words_per_chunks=150) -> list[dict]:
    chunks = []
    for page in pages:
        words = page['text'].split()
        for i in range(0, len(words), words_per_chunks):
            chunks.append({
                'paper_id': page['paper_id'],
                'source_sha256': page['source_sha256'],
                'page': page['page'], 
                'start_word': i,
                'end_word': min(i + words_per_chunks, len(words)),
                'chunk_id' : (
                    f"{page['paper_id']}:{page['source_sha256']}:"
                    f"p{page['page']}:w{i}-{min(i + words_per_chunks, len(words))}"
                    ),
                'text': " ".join(words[i:i+words_per_chunks])
            })
            
    return chunks

In [4]:
pages = extract_pages(Path(DOCUMENT_PATH)/'nemotron3super.pdf')
chunks = chunk_pages(pages)

print(f"Chunk count: {len(chunks)}")
print("Example:")
chunks[0]

Chunk count: 176
Example:


{'paper_id': 'nemotron3super',
 'source_sha256': 'fa7e8a9a1a8f9f716e38e46170371d4bc67c3f6d1ac3e4857c574e911f0e107f',
 'page': 1,
 'start_word': 0,
 'end_word': 150,
 'chunk_id': 'nemotron3super:fa7e8a9a1a8f9f716e38e46170371d4bc67c3f6d1ac3e4857c574e911f0e107f:p1:w0-150',
 'text': '2026-4-15 Nemotron 3 Super: Open, Efficient Mixture-of-Experts Hybrid Mamba-Transformer Model for Agentic Reasoning NVIDIA Abstract. We describe the pre-training, post-training, and quantization of Nemotron 3 Super, a 120 billion (active 12 billion) parameter hybrid Mamba-Attention Mixture-of-Experts model. Nemotron 3 Super is the first model in the Nemotron 3 family to 1) be pre-trained in NVFP4, 2) leverage LatentMoE, a new Mixture-of-Experts architecture that optimizes for both accuracy per FLOP and accuracy per parameter, and 3) include MTP layers for inference acceleration through native speculative decoding. We pre-trained Nemotron 3 Super on 25 trillion tokens followed by post-training using supervised 

### Embedding

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'

model = SentenceTransformer(EMBEDDING_MODEL)

/Users/kritteebenjasilarak/Documents/Project/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1107 20:58:38.942000 46232 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11090.29it/s]


In [6]:
token_lengths = [
    len(model.tokenizer(chunk['text'], truncation=False)["input_ids"])
    for chunk in chunks
]

print("Longest chunk:", max(token_lengths))
print("Model limit:", model.max_seq_length)
print("Over limit:", sum(n > model.max_seq_length for n in token_lengths))

Longest chunk: 455
Model limit: 512
Over limit: 0


In [7]:
texts = [chunk['text'] for chunk in chunks]
p_embeddings = model.encode(texts)
print(p_embeddings.shape)

(176, 384)


### Search

In [8]:
def search(query: str, model, chunks: list, passage_embeddings: list, top_k= 3) -> list:
    if len(query.strip()) < 1:
        raise ValueError("Query must not be blank")
    if top_k < 1:
        raise ValueError("top_k must be at least 1")
    if len(chunks) != len(passage_embeddings):
        raise ValueError("Each chunk must have one passage embedding")
    if len(chunks) == 0:
        return []
    q_embeddings = model.encode(['Represent this sentence for searching relevant passages: '
                                + query])
    cosine_sim = cosine_similarity(q_embeddings, passage_embeddings)
    return sorted([(cosine_sim[0][i], chunks[i]) 
                   for i in range(len(chunks))], 
                   key=lambda x:x[0], 
                   reverse=True)[:top_k]

In [9]:
queries = ['What is the model dimension of Nemotron 3 Super?', 'What is Reinforcemnet learning phase of Nemotron 3 Super post-training?']

for q in queries:
    results = search(q,  model, chunks, p_embeddings, 3)
    print(f'Question: {q}')
    for i in range(len(results)):
        print(f"Score: {results[i][0]}")
        print(f"Page: {results[i][1]['page']}")
        print(f"Text: {results[i][1]['text']}\n")
    

Question: What is the model dimension of Nemotron 3 Super?
Score: 0.8001698851585388
Page: 3
Text: Attention architecture, but Nemotron 3 Super is the first model to scale sparsely using LatentMoE layers rather than standard MoE layers. Configuration Nemotron 3 Super 120B-A12B Base Total Layers 88 Model Dimension 4096 Q-Heads (𝑛𝑞) 32 KV-Heads (𝑛𝑘𝑣) 2 Head Dimension 128 Mamba State Dimension 128 Mamba Groups 8 Mamba Heads 128 Mamba Head Dimension 64 Expert Hidden Dimension 2688 Shared Expert Intermediate Size 5376 Total Experts per Layer 512 Top-𝑘(Activated Experts) 22 MoE Latent Size 1024 MTP layers (shared weight) 2 Table 1 | Nemotron 3 Super Architectural Dimensions. The model employs a hybrid Mamba-2 and MoE design with strategic global attention layers to optimize the balance between sequence modeling performance and inference throughput. 3

Score: 0.7550971508026123
Page: 32
Text: 120B and is competitive to Qwen 3.5 122B on some harnesses. Chat and Instruction Following Capabiliti

### Five-question development benchmark

Evidence was selected from the PDF before running these five searches. Page numbers are 1-based PDF pages. This small development set includes the model-dimension question already used while building search.

Page Hit@3 measures whether any of the top three chunks comes from a labeled evidence page. A matching page can still contain an irrelevant chunk, so inspect the returned text before judging answer support. This set contains five answerable questions and measures retrieval only.

Run the earlier extraction, model, embedding, and search cells first. Rerun this section after changing retrieval settings. Keep words_per_chunk in the report aligned with the chunking call.


In [10]:
benchmark = [
    {
        "id": "dev-01",
        "question": "What is the model dimension of Nemotron 3 Super?",
        "reference_answer": "4096.",
        "evidence_pages": [3],
        "evidence_section": "Table 1",
        "evidence_anchor": "Model Dimension 4096",
    },
    {
        "id": "dev-02",
        "question": "What average acceptance length does Nemotron 3 Super achieve on SPEED-Bench with a draft length of seven?",
        "reference_answer": "3.45 tokens per verification step.",
        "evidence_pages": [6],
        "evidence_section": "Speculative Decoding Performance",
        "evidence_anchor": "highest overall average acceptance length (3.45)",
    },
    {
        "id": "dev-03",
        "question": "How many conversations and domains does the specialized conversational tool-use synthetic pipeline produce?",
        "reference_answer": "279,116 conversations across 838 domains.",
        "evidence_pages": [24],
        "evidence_section": "Conversational Tool Use",
        "evidence_anchor": "279,116 conversations across 838 domains",
    },
    {
        "id": "dev-04",
        "question": "How many environments and RL datasets are used in multi-environment RL training?",
        "reference_answer": "21 environments and 37 RL datasets.",
        "evidence_pages": [27],
        "evidence_section": "RLVR Data",
        "evidence_anchor": "21 environments and 37 different RL datasets",
    },
    {
        "id": "dev-05",
        "question": "Which model initializes the GenRM used for reinforcement learning from human feedback?",
        "reference_answer": "Qwen3-235B-A22B-Thinking-2507.",
        "evidence_pages": [28],
        "evidence_section": "3.2.3. Stage 3",
        "evidence_anchor": "Qwen3-235B-A22B-Thinking-2507 as the initialization",
    },
]


In [11]:
import hashlib
import json
import platform
import sys
from pathlib import Path

corpus_path = Path(DOCUMENT_PATH) / "nemotron3super.pdf"
page_texts = {page['page']: " ".join(page['text'].split()) for page in pages}

# Validate the evidence locations before retrieving.
for case in benchmark:
    assert any(
        case["evidence_anchor"] in page_texts[page_number]
        for page_number in case["evidence_pages"]
    ), f"Evidence anchor missing for {case['id']}"

benchmark_results = []
for case in benchmark:
    hits = search(case["question"], model, chunks, p_embeddings, top_k=3)
    retrieved = [
        {"rank": rank, "page": chunk['page'], "score": float(score)}
        for rank, (score, chunk) in enumerate(hits, start=1)
    ]
    page_hit = any(hit["page"] in case["evidence_pages"] for hit in retrieved)
    benchmark_results.append({
        "id": case["id"],
        "question": case["question"],
        "reference_answer": case["reference_answer"],
        "evidence_pages": case["evidence_pages"],
        "retrieved": retrieved,
        "page_hit_at_3": page_hit,
    })
    retrieved_pages = [hit["page"] for hit in retrieved]
    print(f"{case['id']} | evidence {case['evidence_pages']} | retrieved {retrieved_pages} | hit {page_hit}")

hit_count = sum(result["page_hit_at_3"] for result in benchmark_results)
benchmark_report = {
    "benchmark_version": "nemotron-dev-v1",
    "corpus_sha256": hashlib.sha256(corpus_path.read_bytes()).hexdigest(),
    "benchmark_sha256": hashlib.sha256(
        json.dumps(benchmark, sort_keys=True).encode()
    ).hexdigest(),
    "model": EMBEDDING_MODEL,
    "query_prefix": "Represent this sentence for searching relevant passages: ",
    "words_per_chunk": 150,
    "chunk_count": len(chunks),
    "top_k": 3,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "page_hit_at_3": hit_count / len(benchmark_results),
    "results": benchmark_results,
}
print(f"Page Hit@3: {hit_count}/{len(benchmark_results)} ({benchmark_report['page_hit_at_3']:.0%})")


dev-01 | evidence [3] | retrieved [3, 32, 2] | hit True
dev-02 | evidence [6] | retrieved [7, 6, 32] | hit True
dev-03 | evidence [24] | retrieved [24, 25, 24] | hit True
dev-04 | evidence [27] | retrieved [27, 26, 26] | hit True
dev-05 | evidence [28] | retrieved [28, 28, 24] | hit True
Page Hit@3: 5/5 (100%)


### Chroma DB

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient('../data/')
collection = chroma_client.get_or_create_collection(name='collection', 
                                metadata={"hnsw:space": "cosine",
                                          "embedding_model": EMBEDDING_MODEL,
                                          "words_per_chunk": 150,
                                          "pipeline_version": "1"})

In [91]:
def validate_collection(collection, expected_metadata: dict) -> None:
    metadata = collection.metadata or {}
  
    for field in expected_metadata:
        if metadata.get(field, "<missing>") != expected_metadata[field]:
            raise ValueError(
                            f"{field}: expected {expected_metadata[field]!r}, "
                            f"stored {metadata.get(field, '<missing>')!r}"
                        )

In [95]:
validate_collection(collection, 
            expected_metadata={
                "embedding_model": EMBEDDING_MODEL,
                "words_per_chunk": 150,
                "pipeline_version": "1"
            })


In [ ]:
def upsert_paper(collection, paper_id, chunks, embeddings) -> None:
    validate_collection(collection, expected_metadata={
        "embedding_model": EMBEDDING_MODEL,
        "words_per_chunk": 150,
        "pipeline_version": "1",
    })
    if chunks == []:
        raise ValueError("chunks must have at least 1 element")
    if len(chunks) != len(embeddings):
        raise ValueError("chunks and embeddings must have the same length")
    for chunk in chunks:
        if paper_id != chunk["paper_id"]:
            raise ValueError("Chunk's paper_id does not match the given paper_id")

    old_ids = set(collection.get(where={"paper_id": paper_id})["ids"])
    new_ids = set()
    for i, chunk in enumerate(chunks):
        collection.upsert(
            ids=chunk["chunk_id"],
            embeddings=embeddings[i],
            documents=chunk["text"],
            metadatas={
                "paper_id": chunk["paper_id"],
                "source_sha256": chunk["source_sha256"],
                "page": chunk["page"],
                "start_word": chunk["start_word"],
                "end_word": chunk["end_word"],
            },
        )
        new_ids.add(chunk["chunk_id"])

    stale_ids = old_ids - new_ids
    for chunk_id in stale_ids:
        collection.delete(ids=chunk_id)


In [116]:
upsert_paper(collection=collection,
             paper_id=chunks[0]["paper_id"],
             chunks=chunks,
             embeddings=p_embeddings)

In [14]:
instruction = "Represent this sentence for searching relevant passages: "
query_embedding = model.encode([instruction + queries[0]])

chroma_results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
)
search_results = search(query=queries[0], 
                        model = model, 
                        chunks=chunks, 
                        passage_embeddings=p_embeddings, 
                        top_k=3)

#### Compare chroma and search function

In [15]:
for i in range(3):
    print(f"Rank: {i+1}")
    print(f"Chroma Score: {1 - chroma_results['distances'][0][i]}" )
    print(f"Search Score: {search_results[i][0]}")
    print(f"Agree: {chroma_results['ids'][0][i] == search_results[i][1]['chunk_id']}")

Rank: 1
Chroma Score: 0.8001698851585388
Search Score: 0.8001698851585388
Agree: True
Rank: 2
Chroma Score: 0.7550974488258362
Search Score: 0.7550971508026123
Agree: True
Rank: 3
Chroma Score: 0.7542816400527954
Search Score: 0.7542815208435059
Agree: True


In [ ]:
def search_chroma(query: str, model, collection, top_k=3):
    if len(query.strip()) < 1:
            raise ValueError("Query must not be blank")
    if top_k < 1:
        raise ValueError("top_k must be at least 1")
    
    q_embeddings = model.encode(['Represent this sentence for searching relevant passages: '
                                + query])
    hits = collection.query(
            query_embeddings=q_embeddings,
            n_results=top_k)
    return [{"rank": i + 1,
                "chunk_id": hits['ids'][0][i],
                "score": float(1 - hits['distances'][0][i]),
                "text": hits['documents'][0][i],
                "metadata": {
                    "paper_id": hits['metadatas'][0][i]['paper_id'],
                    "source_sha256": hits['metadatas'][0][i]['source_sha256'],
                    "page": hits['metadatas'][0][i]['page'],
                    "start_word": hits['metadatas'][0][i]['start_word'],
                    "end_word": hits['metadatas'][0][i]['end_word']
                }} 
            for i in range(len(hits["ids"][0]))]
        

In [47]:
for case in benchmark:
    hits = search_chroma(case['question'], model, collection, top_k=3)
    baseline_hits = search(
    case["question"], model, chunks, p_embeddings, top_k=3)
    chroma_ids = [hit["chunk_id"] for hit in hits]
    baseline_ids = [chunk["chunk_id"] for _, chunk in baseline_hits]

    print("Same chunk ranking:", chroma_ids == baseline_ids)
    retrieved_pages = [hit['metadata']['page'] for hit in hits]
    page_hit = case['evidence_pages'][0] in retrieved_pages
    print(f"{case['id']} | evidence {case['evidence_pages']} | retrieved {retrieved_pages} | hit {page_hit}")

Same chunk ranking: True
dev-01 | evidence [3] | retrieved [3, 32, 2] | hit True
Same chunk ranking: True
dev-02 | evidence [6] | retrieved [7, 6, 32] | hit True
Same chunk ranking: True
dev-03 | evidence [24] | retrieved [24, 25, 24] | hit True
Same chunk ranking: True
dev-04 | evidence [27] | retrieved [27, 26, 26] | hit True
Same chunk ranking: True
dev-05 | evidence [28] | retrieved [28, 28, 24] | hit True
